### Allscripts Sunrise (SCM) Drug Catalog Probe

Run this notebook before requesting new SCM medication tables.
It checks whether the current catalog already contains the missing medication master / formulary / RxNorm bridge fields we need to populate SCM `drug_exposure`, `drug_era`, and `dose_era`.

In [0]:
%sql
SELECT table_catalog, table_schema, table_name
FROM _exponent.information_schema.tables
WHERE UPPER(table_name) RLIKE 'SXA|MED|DRUG|RXNORM|FORMULARY|GENERIC|NDC|PHARM'
  AND (
    UPPER(table_schema) LIKE '%ALLSCRIPTS%SCM%'
    OR UPPER(table_schema) LIKE '%SCM%'
  )
ORDER BY table_schema, table_name;

In [0]:
%sql
SELECT table_schema, table_name, column_name, data_type
FROM _exponent.information_schema.columns
WHERE table_name IN (
  'dbo_cv3order',
  'dbo_cv3medicationextension',
  'dbo_sxammgenericitem'
)
ORDER BY table_name, ordinal_position;

In [0]:
%sql
SELECT table_schema, table_name, column_name, data_type
FROM _exponent.information_schema.columns
WHERE (
    UPPER(column_name) RLIKE 'RXNORM|NDC|GENERICITEM|FORMULARY|MEDICATION|DRUG|PHARM|SYNONYM|TRADE|BRAND'
    OR UPPER(table_name) RLIKE 'RXNORM|NDC|FORMULARY|GENERICITEM|MEDICATION|DRUG|PHARM'
  )
  AND (
    UPPER(table_schema) LIKE '%ALLSCRIPTS%SCM%'
    OR UPPER(table_schema) LIKE '%SCM%'
  )
ORDER BY table_schema, table_name, column_name
LIMIT 500;

In [0]:
%sql
SELECT COUNT(*) AS sxammgenericitem_rows,
       SUM(CASE WHEN RxNormCode IS NOT NULL AND TRIM(CAST(RxNormCode AS STRING)) <> '' THEN 1 ELSE 0 END) AS with_rxnorm
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem;

In [0]:
%sql
SELECT *
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem
LIMIT 100;

In [0]:
%sql
SELECT
  COUNT(*) AS medext_rows,
  SUM(CASE WHEN PrescriptionGenericItemID IS NOT NULL THEN 1 ELSE 0 END) AS with_generic_item_id,
  COUNT(DISTINCT PrescriptionGenericItemID) AS distinct_generic_item_ids,
  SUM(CASE WHEN OrderRouteCode IS NOT NULL AND TRIM(CAST(OrderRouteCode AS STRING)) <> '' THEN 1 ELSE 0 END) AS with_route_code
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension;

In [0]:
%sql
SELECT
  PrescriptionGenericItemID,
  COUNT(*) AS row_count
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension
WHERE PrescriptionGenericItemID IS NOT NULL
GROUP BY PrescriptionGenericItemID
ORDER BY row_count DESC
LIMIT 100;

In [0]:
%sql
SELECT
  medext.PrescriptionGenericItemID,
  COUNT(*) AS order_rows,
  MAX(gi.RxNormCode) AS sample_rxnorm
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON CAST(medext.PrescriptionGenericItemID AS STRING) = CAST(gi.DrugID AS STRING)
GROUP BY medext.PrescriptionGenericItemID
ORDER BY order_rows DESC
LIMIT 100;

In [0]:
%sql
SELECT
  COUNT(*) AS joined_rows,
  SUM(CASE WHEN gi.DrugID IS NOT NULL THEN 1 ELSE 0 END) AS with_generic_item_match,
  SUM(CASE WHEN gi.RxNormCode IS NOT NULL AND TRIM(CAST(gi.RxNormCode AS STRING)) <> '' THEN 1 ELSE 0 END) AS with_rxnorm_after_join
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON CAST(medext.PrescriptionGenericItemID AS STRING) = CAST(gi.DrugID AS STRING);

In [0]:
%sql
SELECT
  gi.RxNormCode,
  COUNT(*) AS generic_item_rows,
  MAX(rxnorm_source.concept_id) AS rxnorm_source_concept_id,
  MAX(rxnorm_standard.concept_id) AS mapped_standard_concept_id
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
LEFT JOIN _exponent.omop.concept rxnorm_source
  ON rxnorm_source.concept_code = TRIM(CAST(gi.RxNormCode AS STRING))
 AND rxnorm_source.vocabulary_id IN ('RxNorm', 'RxNorm Extension')
 AND rxnorm_source.domain_id = 'Drug'
 AND rxnorm_source.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept_relationship maps_to
  ON maps_to.concept_id_1 = rxnorm_source.concept_id
 AND maps_to.relationship_id = 'Maps to'
 AND maps_to.invalid_reason IS NULL
LEFT JOIN _exponent.omop.concept rxnorm_standard
  ON rxnorm_standard.concept_id = maps_to.concept_id_2
 AND rxnorm_standard.standard_concept = 'S'
 AND rxnorm_standard.domain_id = 'Drug'
 AND rxnorm_standard.invalid_reason IS NULL
WHERE gi.RxNormCode IS NOT NULL
  AND TRIM(CAST(gi.RxNormCode AS STRING)) <> ''
GROUP BY gi.RxNormCode
ORDER BY generic_item_rows DESC
LIMIT 100;

In [0]:
%sql
SELECT
  ord.Name,
  ord.IDCode,
  medext.PrescriptionGenericItemID,
  gi.DrugID AS generic_item_id,
  gi.RxNormCode,
  COUNT(*) AS row_count
FROM _exponent._bronze_allscripts_scm_prod_01.dbo_cv3order ord
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_cv3medicationextension medext
  ON medext.GUID = ord.GUID
LEFT JOIN _exponent._bronze_allscripts_scm_prod_01.dbo_sxammgenericitem gi
  ON CAST(medext.PrescriptionGenericItemID AS STRING) = CAST(gi.DrugID AS STRING)
WHERE ord.Active = TRUE
  AND ord.TypeCode = 'Medication'
GROUP BY ord.Name, ord.IDCode, medext.PrescriptionGenericItemID, gi.DrugID, gi.RxNormCode
ORDER BY row_count DESC
LIMIT 100;

In [0]:
%sql
SELECT table_schema, table_name, column_name
FROM _exponent.information_schema.columns
WHERE (
    UPPER(column_name) RLIKE 'PRESCRIPTIONGENERICITEMID|RXNORMCODE|NDC|ITEMID|FORMULARY|MEDICATIONID|DRUGID'
  )
  AND (
    UPPER(table_schema) LIKE '%ALLSCRIPTS%SCM%'
    OR UPPER(table_schema) LIKE '%SCM%'
  )
ORDER BY table_schema, table_name, column_name;